<div style="text-align: center;">
  <h1 style="text-align: center;">Reto 2 — Asistente de Guías de Viaje</h1>
  <h3 style="text-align: center;">Master IA &amp; Data Science · EBIS Business School · Bloque B</h3>
  <p style="text-align: center;">Workshop: Prototipado de Agentes</p>
</div>

## Caso

Un viajero está planificando su próximo viaje y pregunta desde el móvil:

> *"¿Cuántos días puedo quedarme en Tailandia sin visado siendo español?"*

El agente debe **buscar en las guías de viaje internas**, responder en lenguaje natural y **citar la fuente** (guía + página).

## Guías disponibles en `data/manuals/`

- `guia_japon.pdf` — Guía completa de Japón (9 páginas)
- `guia_tailandia.pdf` — Guía completa de Tailandia (9 páginas)
- `guia_portugal.pdf` — Guía completa de Portugal (9 páginas)

## Arquitectura objetivo

```
Usuario  →  Agente (Agents SDK)  →  tool: search_kb     →  Chroma (vector store)
                                  →  tool: fetch_section →  PDFs en data/manuals/
                                  ↓
                          Respuesta tipada (texto + citas)
```

## Criterios mínimos para superar el reto

- ✅ Usa la tool `search_kb` al menos una vez por pregunta.
- ✅ La salida es un objeto **tipado** con citas (`fuente` + `página`).
- ✅ Al menos un **guardrail** activo (input o output).
- ✅ Pasa **3 de las 5 preguntas** del eval set (`tests/cases.yaml`).

## Reglas del Reto

- **Individual o en parejas**. **60 minutos**.
- Vibe coding permitido y recomendado.
- Sigue el orden de los TODO. No saltes pasos.
- Mejor citation rate gana la ronda.

## Setup

Instala dependencias y configura el event loop para Jupyter.

In [ ]:
!uv pip install --system -q --upgrade openai-agents==0.4.1 nest_asyncio==1.6.0 "chromadb>=0.5.20" pypdf==4.3.1 pyyaml==6.0.2 "typing_extensions>=4.13.0"

# ⚠️ Tras instalar/actualizar paquetes, REINICIA el kernel (Kernel → Restart Kernel)
# antes de ejecutar las siguientes celdas.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os
from pathlib import Path

MANUALS_DIR = Path("data/manuals")
CHROMA_DIR = Path("data/chroma")
EVAL_FILE = Path("tests/cases.yaml")

MANUALS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
EVAL_FILE.parent.mkdir(parents=True, exist_ok=True)

print("✅ Setup completado.")
print("Guías esperadas en:", MANUALS_DIR.resolve())
print("Ficheros:", [p.name for p in MANUALS_DIR.glob("*.pdf")])

## TODO 1 — Ingesta de guías de viaje en Chroma

Carga los PDFs de `data/manuals/`, trocéalos en chunks de ~800 caracteres y guárdalos en Chroma con metadatos `{manual, pagina}`.

**Tips**:
- Usa `PdfReader` de `pypdf` para extraer texto página a página.
- Guarda **una entrada por página** para que las citas sean naturales.
- La colección se llamará `guias_viaje`.

In [ ]:
import logging
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)

import chromadb
from pypdf import PdfReader

client = chromadb.Client(
    settings=chromadb.Settings(anonymized_telemetry=False),
)
coleccion = client.get_or_create_collection("guias_viaje")

def ingestar_guias(directorio: Path) -> int:
    """
    TODO 1:
    - Recorre todos los PDFs en `directorio`.
    - Por cada página, extrae texto y añade un documento a `coleccion`.
    - Metadatos requeridos: {"manual": <nombre_fichero>, "pagina": <int>}.
    - id sugerido: f"{nombre_fichero}::{pagina}".
    Devuelve el número total de páginas ingestadas.
    """
    raise NotImplementedError("Implementa la ingesta de guías aquí")

# total = ingestar_guias(MANUALS_DIR)
# print(f"Ingestadas {total} páginas")

## TODO 2 — Tool `search_kb`

La tool principal: busca semánticamente en Chroma y devuelve los top-k chunks con sus metadatos.

**Contrato**:
- Recibe `query: str` y `k: int = 4`.
- Devuelve una lista de dicts con `{texto, manual, pagina}`.
- Los resultados deben ser **objetos tipados** — el agente lee estructura, no texto libre.

In [ ]:
from agents import function_tool
from pydantic import BaseModel

class Fragmento(BaseModel):
    texto: str
    manual: str
    pagina: int

@function_tool
def search_kb(query: str, k: int = 4) -> list[Fragmento]:
    """
    TODO 2:
    Busca los k chunks más relevantes en la colección Chroma `guias_viaje`.
    Devuelve cada chunk como `Fragmento(texto, manual, pagina)`.

    Args:
        query: Pregunta del viajero en lenguaje natural.
        k: Número de fragmentos a devolver (por defecto 4).
    """
    raise NotImplementedError("Implementa search_kb usando la colección de Chroma")

## TODO 3 — Tool `fetch_section`

Tool complementaria: cuando el agente quiera ver el contexto completo de una página de la guía.

**Contrato**:
- Recibe `manual: str` (nombre de fichero) y `pagina: int`.
- Devuelve el texto completo de esa página.
- Si no existe → mensaje de error legible (no excepción).

In [ ]:
@function_tool
def fetch_section(manual: str, pagina: int) -> str:
    """
    TODO 3:
    Lee la página `pagina` (1-indexed) del PDF `data/manuals/{manual}`.
    Devuelve el texto completo o un mensaje de error si no existe.

    Args:
        manual: Nombre del fichero PDF (ej: 'guia_japon.pdf').
        pagina: Número de página (empezando en 1).
    """
    raise NotImplementedError("Implementa fetch_section leyendo el PDF directamente")

## TODO 4 — Salida tipada + Guardrails

El agente debe devolver siempre un objeto `Respuesta` con texto **y** al menos una cita.

**Guardrails requeridos** (mínimo 1):
- **Output**: la respuesta debe incluir al menos una `Cita` no vacía. Si no la hay, fuerza un retry o falla con mensaje claro.
- **Input (opcional)**: rechaza preguntas off-topic (que no sean sobre viajes, destinos o planificación).

In [ ]:
from agents import (
    Agent, output_guardrail, input_guardrail,
    GuardrailFunctionOutput, RunContextWrapper,
)
from pydantic import Field

class Cita(BaseModel):
    manual: str
    pagina: int

class Respuesta(BaseModel):
    texto: str = Field(description="Respuesta en lenguaje natural, 3-5 frases")
    citas: list[Cita] = Field(description="Al menos una cita a una guía de viaje")
    confianza: float = Field(description="Confianza 0-1 basada en la calidad de los fragmentos")

@output_guardrail
async def exigir_citas(
    ctx: RunContextWrapper[None],
    agent: Agent,
    output: Respuesta,
) -> GuardrailFunctionOutput:
    """
    TODO 4.a:
    Comprueba que `output.citas` tiene al menos una entrada con manual y pagina > 0.
    Si no, lanza ValueError con mensaje claro.
    """
    raise NotImplementedError("Implementa el guardrail de citas obligatorias")

# Opcional: input guardrail off-topic
# @input_guardrail
# async def rechazar_offtopic(...): ...

## TODO 5 — Construir el agente

Junta las piezas: instrucciones, `output_type`, tools, guardrails.

In [ ]:
SYSTEM_PROMPT = """
Eres TravelMind, el asistente de viajes de EBIS Business School.
Tu única fuente de verdad son las guías de viaje internas accesibles vía las tools.
Reglas:
1. Antes de responder, llama a `search_kb` con la pregunta del viajero.
2. Si necesitas más contexto, usa `fetch_section` para leer la página completa.
3. Responde en 3-5 frases. Sé concreto y accionable.
4. Incluye SIEMPRE al menos una cita (guía + página). Si no hay fuente, dilo.
5. Si la pregunta no está cubierta por las guías, indícalo claramente.
""".strip()

agente_guias = Agent(
    name="TravelMind Guías",
    instructions=SYSTEM_PROMPT,
    # TODO 5: registra las tools y guardrails creados arriba
    tools=[
        # search_kb,
        # fetch_section,
    ],
    output_type=Respuesta,
    output_guardrails=[
        # exigir_citas,
    ],
)

## TODO 6 — Probar el agente

Ejecuta una primera consulta manual y verifica:
1. ¿Llama a `search_kb`? (verás el print)
2. ¿La respuesta tiene `texto` + al menos una `Cita`?
3. ¿La cita coincide con páginas reales de la guía?

In [ ]:
from agents import Runner

# TODO 6: prueba una pregunta real sobre las guías
PREGUNTA = "¿Cuántos días puede quedarse un español en Tailandia sin visado?"

# resultado = Runner.run_sync(agente_guias, PREGUNTA)
# r = resultado.final_output
# print("📝", r.texto)
# print("📑 Citas:")
# for c in r.citas:
#     print(f"   - {c.manual} · p{c.pagina}")
# print("🔎 Confianza:", r.confianza)

## TODO 7 — Eval set

Carga `tests/cases.yaml` y corre el agente contra las 5 preguntas. Para cada caso, comprueba que la respuesta contiene el `keyword` esperado y que la cita es a una guía real.

**Objetivo del Reto**: pasar al menos **3 de 5** casos.

Quien obtenga mayor *citation rate* (citas correctas / total) gana la ronda.

In [ ]:
import yaml

def correr_eval():
    """
    TODO 7:
    - Lee `tests/cases.yaml`.
    - Por cada caso: ejecuta el agente con la pregunta.
    - Comprueba que `keyword` está presente (case-insensitive) en `r.texto`.
    - Comprueba que `r.citas` no está vacío y que el manual citado existe.
    - Imprime un resumen: pasados/total + citation rate.
    """
    casos = yaml.safe_load(EVAL_FILE.read_text())
    pasados = 0
    citas_correctas = 0
    for caso in casos:
        # TODO: implementa la evaluación
        pass
    print(f"Pasados: {pasados}/{len(casos)}")
    print(f"Citation rate: {citas_correctas}/{len(casos)}")

# correr_eval()

## Showcase · 3 minutos por persona o pareja

Cuando termine el tiempo:

1. Demo en vivo de una pregunta dura (la que mejor responda tu agente).
2. Una pregunta donde **falle** — y por qué crees que falla.
3. Una mejora que tienes en mente para la siguiente iteración.

Criterios de evaluación:

| Criterio | Detalle |
|---|---|
| ✅ Funciona | Responde 3/5 del eval set |
| 📑 Cita | Cada respuesta lleva guía + página, real, no inventada |
| 🛡 Robusto | El guardrail rechaza off-topic o respuesta sin citas |
| 🏆 Bonus | Menor latencia media + mayor citation rate |